# Automated Land Cover Sampling Strategy
### This notebook performs an end-to-end spatial analysis to generate high-quality training samples from Corine Land Cover (CLC) data and Sentinel-2 NDVI imagery, for the year 2018.
This is the V5 (Fixed Distribution): Approximately 200 polygons sampled across all 28 classes including 4 seasons. 

### inputs : 
1. one raster of NDVI, 
2. one cropped shp of CLC or any local reference layer
### outputs: 
1. Square polygon of of pure samples 
2. Randomly selected up to 60 polygons per class

## Step 0: Environment Setup

In [1]:
# 1. Import libraries
import geopandas as gpd
import pandas as pd
import rasterio
from rasterstats import zonal_stats
import folium
from folium import plugins
import json
import ee
import os
from ipyleaflet import Map, DrawControl
import time
import datetime
import math, requests
from pathlib import Path
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import io
import ipywidgets as widgets
from IPython.display import display, Javascript, HTML, clear_output, Markdown
import numpy as np
from rasterio.mask import mask
from shapely.geometry import Point, box, mapping
from shapely.ops import unary_union, transform
from shapely.validation import make_valid
from sklearn.ensemble import IsolationForest
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import textwrap
import pyproj
import matplotlib.gridspec as gridspec
import matplotlib.patches as patches
import textwrap
import random
from cdsetool.query import query_features
from cdsetool.download import download_features
from cdsetool.credentials import Credentials
import zipfile
from rasterio.warp import reproject, Resampling
from rasterio.crs import CRS
import shutil
from shapely.prepared import prep
import calendar
import warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=FutureWarning)

print ("Libraries imported correctly!")

C:\ProgramData\anaconda3\envs\Cop_t24\lib\site-packages\google\api_core\_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


Libraries imported correctly!


In [2]:
# 2. Initialize Earth Engine and Select bounding box AOI

# Initialize (authenticate once with ee.Authenticate() if needed)
try:
    ee.Initialize()
except Exception:
    ee.Authenticate()
    ee.Initialize()

print ("Earth Engine initialized!")

def mask_s2_clouds(image):

    qa = image.select('QA60')

    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11

    mask = (
        qa.bitwiseAnd(cloud_bit_mask).eq(0)
        .And(
            qa.bitwiseAnd(cirrus_bit_mask).eq(0)
        )
    )

    return (
        image.updateMask(mask)
        .divide(10000)
    )

def get_seasonal_ndvi(aoi, start_date, end_date):

    s2 = (
        ee.ImageCollection(
            "COPERNICUS/S2_SR_HARMONIZED"
        )
        .filterBounds(aoi)
        .filterDate(start_date, end_date)
        .filter(
            ee.Filter.lt(
                "CLOUDY_PIXEL_PERCENTAGE",
                20
            )
        )
        .map(mask_s2_clouds)
    )

    composite = s2.median()

    ndvi = (
        composite.normalizedDifference(
            ["B8", "B4"]
        )
        .rename("NDVI")
    )

    return ndvi


# =========================
# AOI SELECTION
# =========================

mode = input(
    "\nChoose input mode:\n"
    "1 = Enter longitude/latitude\n"
    "2 = Select point on map\n\n"
    "Option: "
).strip()

# -------------------------
# COORDINATE INPUT
# -------------------------
if mode == "1":

    coords = input(
        "\nEnter longitude,latitude "
        "(example Seville: -5.9845,37.3891): "
    )

    POINT_LON, POINT_LAT = map(
        float,
        coords.split(",")
    )

# -------------------------
# MAP CLICK
# -------------------------
elif mode == "2":

    from ipyleaflet import Map, Marker
    from IPython.display import display

    clicked = {}

    m = Map(center=(40, -3), zoom=6)

    marker = Marker(location=(40, -3))
    m.add(marker)

    def handle_click(**kwargs):

        if kwargs.get('type') == 'click':

            latlon = kwargs.get('coordinates')

            marker.location = latlon

            clicked["lat"] = latlon[0]
            clicked["lon"] = latlon[1]

            print(
                f"Selected:"
                f" {latlon[1]:.5f}, {latlon[0]:.5f}"
            )

    m.on_interaction(handle_click)

    display(m)

    input("\nClick map then press ENTER here...")

    POINT_LAT = clicked["lat"]
    POINT_LON = clicked["lon"]

else:

    raise ValueError("Invalid option")

# -------------------------
# SMALL AOI AROUND POINT
# -------------------------
BUFFER_DEG = 0.15

geom_4326 = box(
    POINT_LON - BUFFER_DEG,
    POINT_LAT - BUFFER_DEG,
    POINT_LON + BUFFER_DEG,
    POINT_LAT + BUFFER_DEG
)

geom_wkt = geom_4326.wkt

# -------------------------
# AUTOMATIC TILE DETECTION
# -------------------------
point = ee.Geometry.Point([
    POINT_LON,
    POINT_LAT
])

s2 = (
    ee.ImageCollection(
        "COPERNICUS/S2_SR_HARMONIZED"
    )
    .filterBounds(point)
    .first()
)

TARGET_TILE = s2.get(
    "MGRS_TILE"
).getInfo()

TARGET_YEAR = int(input("Enter processing year (example 2025): ").strip())

print("✅ Using tile:", TARGET_TILE)
print("✅ Using year:", TARGET_YEAR)

Earth Engine initialized!



Choose input mode:
1 = Enter longitude/latitude
2 = Select point on map

Option:  1

Enter longitude,latitude (example Seville: -5.9845,37.3891):  -5.9845,37.3891
Enter processing year (example 2025):  2018


✅ Using tile: 29SQB
✅ Using year: 2018


In [3]:
# 3. Setup Paths
root_dir = r'C:\Users\fdomingues\Cop_t24\CLC_sampling_process_SPAIN'
#input_shp = os.path.join(root_dir, 'CLC_L3_2018_spain.shp')
input_gpkg = r"C:\Users\fdomingues\Cop_t24\CLC_sampling_process_SPAIN\reclass_CLC_28class.gpkg"
CLC_LAYER = "reclassCLCin28Class"
#raster_path = os.path.join(root_dir, 'NDVI_Tile_29SQB_2018_spain.tif')
SEASON_OUTPUT = os.path.join(
    root_dir,
    "seasonal_outputs"
)

# Define sub-folders
folders = ['split_outputs', 'processed_poly', 'merged_reclassified', 'exports']
paths = {folder: os.path.join(root_dir, folder) for folder in folders}
SOURCE_CLASS_FIELD = "clc_l3"   # change if your modified classes are stored in another field
TOP_N_HIGH_AREA = 8              # choose how many highest-area classes to keep
FORCE_INCLUDE = {"310", "325", "330", "331"}
OTHER_CLASS = "999"

# Create all necessary directories
for path in paths.values():
    if not os.path.exists(path):
        os.makedirs(path)

# ----------------------------
# Status system (like flood example)
# ----------------------------
VERBOSE = True
MIN_DISPLAY_SEC = 1
_last_status_time = None

def _wait_for_previous(min_seconds: float):
    global _last_status_time
    if _last_status_time is not None:
        elapsed = time.time() - _last_status_time
        if elapsed < min_seconds:
            time.sleep(min_seconds - elapsed)

def status(msg: str, *, done: bool=False, min_seconds: float = MIN_DISPLAY_SEC):
    if not VERBOSE:
        return
    _wait_for_previous(min_seconds)
    with status_area:
        status_area.clear_output(wait=True)
        style = "color:#333;"
        if done:
            style = "color:#2e7d32;"
        display(HTML(
            f'<div style="font-family:Segoe UI,Arial;font-size:14px;'
            f'padding:6px 8px;border-left:4px solid #888;{style}">{msg}</div>'
        ))
    global _last_status_time
    _last_status_time = time.time()

class StepTimer:
    def __init__(self, start_msg: str, done_label: str = "done", min_seconds: float = MIN_DISPLAY_SEC):
        self.start_msg = start_msg
        self.done_label = done_label
        self.min_seconds = min_seconds
        self.t0 = None
    def __enter__(self):
        self.t0 = time.time()
        status(self.start_msg, done=False, min_seconds=self.min_seconds)
        return self
    def __exit__(self, exc_type, exc, tb):
        dt = time.time() - self.t0
        if exc_type is None:
            status(f"{self.start_msg} — {self.done_label} in {dt:.2f}s", done=True, min_seconds=self.min_seconds)
        else:
            with status_area:
                status_area.clear_output(wait=True)
                display(HTML(
                    f'<div style="font-family:Segoe UI,Arial;font-size:14px;'
                    f'padding:6px 8px;border-left:4px solid #c62828; color:#c62828;">'
                    f'⚠️ Error during: {self.start_msg}<br><code>{exc}</code></div>'
                ))

print("Libraries imported and directories initialized.")

Libraries imported and directories initialized.


## Step 1: Class Splitting

We split the dataset into individual files based on the class.

In [4]:
# 4. Change the values for reclassification based on the need
def log_time(msg, start=None):
    now = time.time()
    ts = datetime.now().strftime("%H:%M:%S")
    if start:
        print(f"[{ts}] {msg} — {(now - start):.1f}s")
    else:
        print(f"[{ts}] {msg}")
    return now


# Dedicated output area
status_area = widgets.Output()
display(status_area)

with StepTimer("Reading updated 28-class CLC and exporting split classes..."):
    gdf_main = gpd.read_file(input_gpkg, layer=CLC_LAYER)

    # Use the final class field directly
    gdf_main["clc_l3"] = gdf_main["clc_l3"].astype(str).str.strip()

    # Remove null / empty classes
    gdf_main = gdf_main[gdf_main["clc_l3"].notna()].copy()
    gdf_main = gdf_main[gdf_main["clc_l3"] != ""].copy()

    print("Unique classes found:")
    print(sorted(gdf_main["clc_l3"].unique()))

    # OPTIONAL: clean old split outputs to avoid mixing old files
    for f in os.listdir(paths["split_outputs"]):
        if f.endswith(".gpkg") or f.endswith(".shp"):
            try:
                os.remove(os.path.join(paths["split_outputs"], f))
            except:
                pass

    # Export one file per final class
    for clc_val, subset in gdf_main.groupby("clc_l3"):
        out_file = os.path.join(paths["split_outputs"], f"CLC_Class_{clc_val}.gpkg")
        subset.to_file(out_file, driver="GPKG")
        status(f"Exported Class: {clc_val}")

    print("Done. Split outputs written to:")
    print(paths["split_outputs"])

Output()

Unique classes found:
['11', '12', '13', '14', '211', '212', '213', '221', '222', '223', '231', '24', '244', '310', '311', '312', '313', '321', '323', '324', '325', '330', '331', '411', '421', '422', '51', '52']
Done. Split outputs written to:
C:\Users\fdomingues\Cop_t24\CLC_sampling_process_SPAIN\split_outputs


## Step 2: Feature extraction (NDVI + optional S2)

We compute NDVI stats using your NDVI raster; if you provide a local S2 band stack, we also compute per‑band standard deviations for additional purity signals.

In [6]:
# 5. Feature extraction + AI-assisted ranking + NDVI ----

# =========================
# CONFIG
# =========================
USERNAME = "francisco.domingues@uab.es"
PASSWORD = "19@Greenday81"

credentials = Credentials(USERNAME, PASSWORD)

export_dir = os.path.join(r"C:\Users\fdomingues\cop_t24\CLC_sampling_process_SPAIN", "exports")
os.makedirs(export_dir, exist_ok=True)

winter_end_day = 29 if calendar.isleap(TARGET_YEAR) else 28

SEASONS = {
    "winter": (f"{TARGET_YEAR-1}-12-01", f"{TARGET_YEAR}-02-{winter_end_day:02d}"),
    "spring": (f"{TARGET_YEAR}-03-01", f"{TARGET_YEAR}-05-31"),
    "summer": (f"{TARGET_YEAR}-06-01", f"{TARGET_YEAR}-08-31"),
    "autumn": (f"{TARGET_YEAR}-09-01", f"{TARGET_YEAR}-11-30")
}

# AOI already defined in Cell 2
#print("✅ Using tile:", TARGET_TILE)

# -------------------------
# SEASON PROCESSING MODE
# -------------------------
SEASON_MODE = input(
    "Enter season mode ('all', 'winter', 'spring', 'summer', 'autumn', or 'none'): "
).strip().lower()

aliases = {
    "w": "winter",
    "sp": "spring",
    "su": "summer",
    "a": "autumn"
}

SEASON_MODE = aliases.get(SEASON_MODE, SEASON_MODE)

valid_modes = {"all", "winter", "spring", "summer", "autumn", "none"}

if SEASON_MODE not in valid_modes:
    raise ValueError(
        "Invalid season mode. Use one of: all, winter, spring, summer, autumn, none"
    )

print("✅ Season mode:", SEASON_MODE)

# =========================
# FUNCTION: compute NDVI
# =========================
def compute_ndvi_array(b4_path, b8_path, scl_path, season):
    
    with rasterio.open(b4_path) as red:
        red_arr = red.read(1).astype(np.float32)
        profile = red.profile.copy()
        transform = red.transform

    with rasterio.open(b8_path) as nir:
        nir_arr = nir.read(1).astype(np.float32)

    with rasterio.open(scl_path) as scl_src:
        scl_arr = scl_src.read(1)

        # ✅ simple upscaling (20m → 10m)
        scale = int(red_arr.shape[0] / scl_arr.shape[0])
        scl_resampled = scl_arr.repeat(scale, axis=0).repeat(scale, axis=1)
        scl_resampled = scl_resampled[:red_arr.shape[0], :red_arr.shape[1]]

    # cloud mask
    if season == "winter":
    
        # less aggressive in winter
        cloud_mask = np.isin(
            scl_resampled,
            [3, 9, 10]
        )
    
    else:
    
        cloud_mask = np.isin(
            scl_resampled,
            [3, 8, 9, 10]
        )
        
    ndvi = (nir_arr - red_arr) / (nir_arr + red_arr + 1e-6)
    ndvi[cloud_mask] = np.nan

    return ndvi, profile
# =========================
# MAIN LOOP
# =========================
    
MAX_SCENES = 16   # try 5–10
used_scenes_log = []
season_scene_dates = {}

for season, (start_date, end_date) in SEASONS.items():

    if SEASON_MODE == "none":
        print("ℹ️ NDVI processing skipped (season mode = none)")
        break

    if SEASON_MODE != "all" and season != SEASON_MODE:
        continue

    print(f"\n🔍 Processing {season}...")
        
    # -------------------------
    # Query Sentinel-2
    # -------------------------
    features = list(query_features(
        "SENTINEL-2",
        {
            "contentDateStartGe": start_date,
            "contentDateStartLe": end_date,
            "productType": "S2MSI2A",
            "cloudCover": [0, 40],
            "geometry": geom_wkt
        }
    ))

    if len(features) == 0:
        print(f"❌ No data for {season}")
        continue

    # sort by cloud cover
    def get_cloud_cover(f):
        try:
            return f.get("cloudCover", 100)
        except:
            return 100
    
    
    features_sorted = sorted(features, key=get_cloud_cover)
    features_sorted = features_sorted[:MAX_SCENES]

    # keep only scenes from that tile
    candidates = features_sorted    
    
    ndvi_stack = []
    profile_ref = None

    # -------------------------
    # PROCESS EACH SCENE
    # -------------------------
    for i, candidate in enumerate(candidates):

        print(f"   ↳ Scene {i+1}/{len(candidates)}")
        scene_start = time.time()
        out_dir = os.path.join(export_dir, f"{season}_tmp_{i}")
        os.makedirs(out_dir, exist_ok=True)
        
        t0 = time.time()

        list(download_features(
            [candidate],
            out_dir,
            {"credentials": credentials}
        ))

        print(
            f"Download took "
            f"{time.time()-t0:.1f} sec"
        )    

        t0 = time.time()

        # extract ZIP
        for f in os.listdir(out_dir):
            if f.endswith(".zip"):
                zip_path = os.path.join(out_dir, f)
                
                with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                    zip_ref.extractall(out_dir)
                
                # ✅ DELETE zip immediately
                os.remove(zip_path)

        print(
            f"Unzip took "
            f"{time.time()-t0:.1f} sec"
        )

        # find SAFE folder
        safe_dirs = [
            os.path.join(out_dir, d)
            for d in os.listdir(out_dir)
            if d.endswith(".SAFE")
        ]

        if not safe_dirs:
            continue

        safe_dir = safe_dirs[0]
        
        safe_name = os.path.basename(safe_dir)
        
        safe_tile = safe_name.split("_")[5]

        acq_token = safe_name.split("_")[2]   # e.g. 20241203T110419
        acq_date = f"{acq_token[:4]}-{acq_token[4:6]}-{acq_token[6:8]}"
        acq_datetime = (
            f"{acq_token[:4]}-{acq_token[4:6]}-{acq_token[6:8]} "
            f"{acq_token[9:11]}:{acq_token[11:13]}:{acq_token[13:15]}"
        )

        print(f"   SAFE tile: {safe_tile}")
        
        if safe_tile != f"T{TARGET_TILE}":
        
            print("   ⚠️ Wrong tile → skipping")
        
            continue     
            
        
        # find bands
        b4 = b8 = scl = None

        for root, _, files in os.walk(safe_dir):
            for f in files:
                full_path = os.path.join(root, f)

                if "R10m" in root:
                    if "_B04_10m.jp2" in f:
                        b4 = full_path
                    elif "_B08_10m.jp2" in f:
                        b8 = full_path

                if "R20m" in root and "_SCL_20m.jp2" in f:
                    scl = full_path

        if not (b4 and b8 and scl):
            print("   ❌ Missing bands → skipping")
            continue

        # NDVI
        ndvi, profile = compute_ndvi_array(
            b4,
            b8,
            scl,
            season
        )

        print(
            "NDVI:",
            round(time.time()-scene_start,1),
            "sec"
        )

        if np.isnan(ndvi).all():
            print("   ❌ all pixels masked")
            continue

        ndvi_stack.append(ndvi)
        
        used_scenes_log.append({
            "season": season,
            "tile": safe_tile,
            "safe_name": safe_name,
            "acquisition_date": acq_date,
            "acquisition_datetime": acq_datetime
        })
        
        season_scene_dates.setdefault(season, [])
        season_scene_dates[season].append(acq_date)

        if profile_ref is None:
            profile_ref = profile

        print(f"   ✅ valid pixels: {np.sum(np.isfinite(ndvi))/ndvi.size:.2f}")

    # -------------------------
    # COMPOSITE
    # -------------------------
    if len(ndvi_stack) == 0:
        print(f"❌ No valid NDVI for {season}")
        continue

    print(f"✅ Combining {len(ndvi_stack)} scenes...")

    stack = np.stack(ndvi_stack)
    ndvi_median = np.nanmedian(stack, axis=0)

    ndvi_out = os.path.join(export_dir, f"ndvi_{season}.tif")

    profile_ref.update(
        driver="GTiff",
        dtype=rasterio.float32,
        count=1,
        nodata=-9999,
        compress="lzw"
    )

    ndvi_median[np.isnan(ndvi_median)] = -9999

    with rasterio.open(ndvi_out, 'w', **profile_ref) as dst:
        dst.write(ndvi_median.astype(np.float32), 1)

    print(f"✅ FINAL NDVI exported: {ndvi_out}")
    print("🧹 Cleaning temporary files...")

    for i in range(MAX_SCENES):
        tmp_folder = os.path.join(export_dir, f"{season}_tmp_{i}")
    
        if os.path.exists(tmp_folder):
            shutil.rmtree(tmp_folder, ignore_errors=True)
            print(f"   ✅ Deleted: {tmp_folder}")


scene_log_df = pd.DataFrame(used_scenes_log)

if not scene_log_df.empty:
    scene_log_df = scene_log_df.sort_values(["season", "acquisition_datetime"])
    scene_log_csv = os.path.join(export_dir, "used_s2_scenes_by_season.csv")
    scene_log_df.to_csv(scene_log_csv, index=False)
    print("Saved used scene log:")
    print(scene_log_csv)


# Paths from your script
#root_dir = r'C:\Users\fdomingues\cop_t24\CLC_sampling_process_SPAIN'  # keep as in your file
#raster_ndvi_path = os.path.join(root_dir, 'NDVI_Tile_29SQB_2018_spain.tif')  # NDVI raster (your script)  # ⟵ uses your NDVI input
merged_path = os.path.join(root_dir, 'merged_reclassified', 'Merged_CLC_reclassified_Classes.gpkg')  # ⟵ your merge output
#export_dir = os.path.join(root_dir, 'exports')

# OPTIONAL: local multi-band Sentinel-2 stack (same CRS/resolution as NDVI)
# If you have a 6-band stack (B2,B3,B4,B8,B11,B12), set this path; else leave as None.
raster_s2_stack_path = None  # e.g., os.path.join(root_dir, 'S2_L2A_2018_stack.tif')

# Quality scoring weights (tune if you like)
W = {
    "ndvi_std": 0.35,          # lower is better
    "ndvi_range": 0.15,        # narrower p90-p10 is better
    "s2_band_std": 0.25,       # lower multi-band std is better
    "interior_margin": 0.15,   # more distance to boundary is better
    "iso_forest": 0.10         # anomaly score -> lower is better
}

# Distance threshold for spatial dispersion (meters)
MIN_SPACING = 500

# Default purity gates (adaptive top-up will relax these)
STDEV_LIMITS = [0.06, 0.08, 0.10]  # your original 0.06 first, then looser
#INNER_BUFFERS = [-100, -60, -40, -20]  # meters
#SQUARE_HALF_SIDES = [20, 16, 12, 10]   # meters -> squares 40, 32, 24, 20 m

INNER_BUFFERS = [-60, -40, -20, -10, 0]   # added 0 as a last-resort fallback
SQUARE_HALF_SIDES = [16, 12, 10, 8]       # 32m, 24m, 20m, 16m squares

# Helper: normalize feature to 0..1 within each class (robust to outliers)
def robust_minmax(x):
    q1, q99 = np.nanpercentile(x, [1, 99])
    return np.clip((x - q1) / max(1e-9, (q99 - q1)), 0, 1)

# Greedy spacing filter
def enforce_min_spacing(gdf, min_dist):
    selected = []
    tree = None
    try:
        from shapely.strtree import STRtree
        tree = STRtree(gdf.geometry.values)
    except Exception:
        # fallback without spatial index (slower)
        pass

    for i, geom in enumerate(gdf.geometry.values):
        if not selected:
            selected.append(i)
            continue
        ok = True
        for j in selected:
            if geom.distance(gdf.geometry.values[j]) < min_dist:
                ok = False
                break
        if ok:
            selected.append(i)
    return gdf.iloc[selected]
print("Finishing setting rankings!")

Enter season mode ('all', 'winter', 'spring', 'summer', 'autumn', or 'none'):  none


✅ Season mode: none
ℹ️ NDVI processing skipped (season mode = none)
Finishing setting rankings!


## Step 3: Spatial Filtering (Inner Buffering & Square Creation)

To ensure spectral purity, we:

Apply a -100m inner buffer to avoid "edge effects" near class boundaries.

Generate a 40x40m square (20m radius) training site at the centroid of the safe zone.


In [ ]:
# 6. SEASONAL ADAPTIVE SAMPLING
# Dedicated output area
status_area = widgets.Output()
display(status_area)

with StepTimer("Seasonal adaptative sampling processing..."):
    # 5. SEASONAL ADAPTIVE SAMPLING FROM ORIGINAL CLC POLYGONS
    # ---------------------------------------------------------
    # Run this AFTER seasonal NDVI rasters exist.
    # If Cell 7 already created ndvi_winter.tif / ndvi_spring.tif / ndvi_summer.tif / ndvi_autumn.tif,
    # you can run this directly and skip old Cells 6, 8 and 9.
    
    print("Starting NEW CELL 5: seasonal adaptive sampling...", flush=True)
    
    # ---------------------------------------------------------
    # CONFIG
    # ---------------------------------------------------------
    TARGET_PER_SEASON = 60
    SEASONS_TO_RUN = ["winter", "spring", "summer", "autumn"]   # change if needed , "spring", "summer", "autumn"
    SEASONAL_NDVI = {
        "winter": os.path.join(export_dir, "ndvi_winter.tif"),
        "spring": os.path.join(export_dir, "ndvi_spring.tif"),
        "summer": os.path.join(export_dir, "ndvi_summer.tif"),
        "autumn": os.path.join(export_dir, "ndvi_autumn.tif"),
    }
    
    # Candidate square sizes (full side length in meters)
    SQUARE_SIZES = [100, 40, 20] #16, 10
    
    # Multi-offset deterministic grid
    GRID_OFFSETS = [
        (0.00, 0.00),
        (0.50, 0.00),
        (0.00, 0.50),
        (0.50, 0.50),
        (0.25, 0.25),
        (0.75, 0.75),
    ]
    
    MAX_OVERLAP_FRAC = 0.25
    MAX_RAW_BY_SIZE = {
        100: 150,
        40: 150,
        20: 150,
    } #        20: 600,        16: 800,        12: 1000,        10: 1200,        8: 1500, 
    MAX_NDVI_SHORTLIST = 120 #500
    
    # Score weights
    W_NDVI = 0.80        # closeness to class-season target NDVI
    W_STD  = 0.15        # lower NDVI std is better
    W_SIZE = 0.05        # prefer bigger squares if all else equal

    TEST_CLASSES = {"223"} # "421", "422", "52", "411", "325"
    # ---------------------------------------------------------
    # HELPERS
    # ---------------------------------------------------------
    def keep_valid_geom(geom):
        """
        Very cheap geometry check.
        No repair here.
        """
        if geom is None or geom.is_empty:
            return None
        if geom.geom_type not in ["Polygon", "MultiPolygon"]:
            return None
        return geom
    
    def repair_if_invalid(geom):
        """
        Repair ONLY if invalid.
        Much cheaper than always calling buffer(0).
        """
        if geom is None or geom.is_empty:
            return None
        if geom.geom_type not in ["Polygon", "MultiPolygon"]:
            return None
        if geom.is_valid:
            return geom
        try:
            geom = geom.buffer(0)
        except Exception:
            return None
        if geom is None or geom.is_empty or not geom.is_valid:
            return None
        return geom
        
    def light_clean_gdf(gdf, repair_invalid=False):
        gdf = gdf.copy()
    
        gdf = gdf[gdf.geometry.notna()].copy()
        gdf = gdf[~gdf.geometry.is_empty].copy()
        gdf = gdf[gdf.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
    
        if repair_invalid:
            bad = ~gdf.geometry.is_valid
            if bad.any():
                gdf.loc[bad, "geometry"] = gdf.loc[bad, "geometry"].buffer(0)
                gdf = gdf[gdf.geometry.notna()].copy()
                gdf = gdf[~gdf.geometry.is_empty].copy()
                gdf = gdf[gdf.geometry.is_valid].copy()
    
        return gdf
    
    def overlap_fraction(a, b):
        try:
            if a is None or b is None or a.is_empty or b.is_empty:
                return 0.0
    
            minx1, miny1, maxx1, maxy1 = a.bounds
            minx2, miny2, maxx2, maxy2 = b.bounds
    
            if maxx1 <= minx2 or maxx2 <= minx1 or maxy1 <= miny2 or maxy2 <= miny1:
                return 0.0
    
            inter = a.intersection(b).area
            if inter <= 0:
                return 0.0
    
            denom = min(a.area, b.area)
            if denom <= 0:
                return 1.0
    
            return inter / denom
        except Exception:
            return 1.0
    
    def overlaps_too_much(geom, accepted_geoms, max_frac=0.25):
        for g in accepted_geoms:
            if overlap_fraction(geom, g) > max_frac:
                return True
        return False
    
    def robust_minmax(x):
        x = pd.to_numeric(pd.Series(x), errors="coerce")
        if x.notna().sum() == 0:
            return pd.Series(np.zeros(len(x)), index=x.index)
        q1, q99 = np.nanpercentile(x.dropna(), [1, 99])
        return np.clip((x - q1) / (q99 - q1 + 1e-9), 0, 1)
    
    def polygon_parts(poly):
        if poly is None or poly.is_empty:
            return []
        if poly.geom_type == "Polygon":
            return [poly]
        if poly.geom_type == "MultiPolygon":
            return sorted(list(poly.geoms), key=lambda g: g.area, reverse=True)
        return []

    def generate_candidates_for_source_polygon(src_geom, src_poly_id, cl, season_name,
                                               square_size, max_raw, ndvi_bbox_3035,
                                               raster_ndvi_path, ndvi_crs, ndvi_nodata):
        """
        Generate NDVI-scored candidates for ONE original source polygon.
        Strictly contained in that source polygon.
        """
        src_geom = safe_intersection(src_geom, ndvi_bbox_3035)
        src_geom = keep_valid_geom(src_geom)
        if src_geom is None:
            return gpd.GeoDataFrame(columns=["clc_l3", "season", "square_size", "src_poly_id", "geometry"],
                                    geometry="geometry", crs="EPSG:3035")
    
        squares = generate_grid_candidates(
            class_polygon=src_geom,
            square_size=square_size,
            max_raw=max_raw
        )
    
        if len(squares) == 0:
            return gpd.GeoDataFrame(columns=["clc_l3", "season", "square_size", "src_poly_id", "geometry"],
                                    geometry="geometry", crs="EPSG:3035")
    
        gdf = gpd.GeoDataFrame(
            {
                "clc_l3": [cl] * len(squares),
                "season": [season_name] * len(squares),
                "square_size": [square_size] * len(squares),
                "src_poly_id": [src_poly_id] * len(squares),
            },
            geometry=squares,
            crs="EPSG:3035"
        )
    
        # strict containment in the ORIGINAL source polygon
        gdf = gdf[gdf.geometry.apply(lambda sq: src_geom.covers(sq))].copy()
        if gdf.empty:
            return gdf
    
        gdf = extract_ndvi_stats(gdf, raster_ndvi_path, ndvi_crs, ndvi_nodata)
        return gdf
    

    def generate_grid_candidates(class_polygon, square_size, max_raw, offsets=GRID_OFFSETS):
        """
        Deterministic multi-offset grid of fully inside squares.
        Uses NO repeated geometry repair.
        Assumes class_polygon is already valid enough.
        """
        class_polygon = keep_valid_geom(class_polygon)
        if class_polygon is None:
            return []
    
        half = square_size / 2.0
        fit_polygon = class_polygon.buffer(-half)
    
        if fit_polygon is None or fit_polygon.is_empty:
            return []
    
        fit_polygon_s = fit_polygon.simplify(5, preserve_topology=True)
        class_polygon_s = class_polygon.simplify(5, preserve_topology=True)
    
        if fit_polygon_s is None or fit_polygon_s.is_empty:
            return []
        if class_polygon_s is None or class_polygon_s.is_empty:
            return []
    
        parts = polygon_parts(fit_polygon_s)
        if not parts:
            return []
    
        prepared_cover = prep(class_polygon_s)
        seen = set()
        squares = []
    
        for part in parts:
            minx, miny, maxx, maxy = part.bounds
            prepared_part = prep(part)
    
            for ox, oy in offsets:
                xs = np.arange(minx + ox * square_size, maxx, square_size)
                ys = np.arange(miny + oy * square_size, maxy, square_size)
    
                for y in ys:
                    for x in xs:
                        key = (round(x, 2), round(y, 2), round(square_size, 2))
                        if key in seen:
                            continue
    
                        p = Point(x, y)
                        if not prepared_part.contains(p):
                            continue
    
                        sq = box(x - half, y - half, x + half, y + half)
    
                        if not prepared_cover.covers(sq):
                            continue
    
                        seen.add(key)
                        squares.append(sq)
    
                        if len(squares) >= max_raw:
                            return squares
    
        return squares

        
    def build_class_union_gdf(input_gpkg, layer_name, clc_crs_target="EPSG:3035"):
        """
        Read original CLC polygons and dissolve by class using TRUE class geometry.
        """
        # read fewer columns if possible
        gdf = gpd.read_file(input_gpkg, layer=layer_name)
    
        if "clc_l3" not in gdf.columns:
            source_code_col = "Code_18" if "Code_18" in gdf.columns else None
            if not source_code_col:
                raise KeyError("Could not find CLC code column.")
            gdf["clc_l3"] = gdf[source_code_col].apply(reclassify_clc)
    
        gdf = gdf.dropna(subset=["clc_l3"]).copy()
        gdf["clc_l3"] = gdf["clc_l3"].astype(str)
        gdf = gdf.reset_index(drop=True)
        gdf["src_poly_id"] = gdf.index.astype(str)
    
        if gdf.crs is None:
            raise ValueError("Input CLC has no CRS.")
        if gdf.crs.is_geographic:
            gdf = gdf.to_crs(clc_crs_target)
        elif str(gdf.crs) != clc_crs_target:
            gdf = gdf.to_crs(clc_crs_target)
    
        gdf = light_clean_gdf(gdf, repair_invalid=False)
    
        union_gdf = gdf[["clc_l3", "geometry"]].dissolve(by="clc_l3", as_index=True)
    
        # repair ONLY invalid dissolved unions
        bad = ~union_gdf.geometry.is_valid
        if bad.any():
            union_gdf.loc[bad, "geometry"] = union_gdf.loc[bad, "geometry"].apply(repair_if_invalid)
    
        union_gdf = union_gdf[union_gdf.geometry.notna()].copy()
        union_gdf = union_gdf[~union_gdf.geometry.is_empty].copy()
        union_gdf["clc_l3"] = union_gdf.index.astype(str)
    
        return union_gdf.reset_index(drop=True), gdf
    
    def extract_ndvi_stats(cand_gdf, raster_path, ndvi_crs, ndvi_nodata):
        """
        Extract mean/std NDVI for candidates.
        """
        if cand_gdf.empty:
            return cand_gdf.copy()
    
        cand_ndvi = cand_gdf.to_crs(ndvi_crs)
        zs = zonal_stats(
            cand_ndvi,
            raster_path,
            stats=["mean", "std"],
            nodata=ndvi_nodata
        )
    
        out = cand_gdf.copy()
        out["ndvi_mean"] = pd.to_numeric([z.get("mean", np.nan) for z in zs], errors="coerce")
        out["ndvi_std"]  = pd.to_numeric([z.get("std",  np.nan) for z in zs], errors="coerce")
        out = out[out["ndvi_mean"].notna()].copy()
        out["ndvi_std"] = out["ndvi_std"].fillna(out["ndvi_std"].median())
        return out
    
    def season_class_target(values):
        """
        Season-specific representative NDVI target for one class.
        No hardcoded 'water low / forest high' assumptions.
        We learn the target from that class in that season.
        """
        values = pd.to_numeric(pd.Series(values), errors="coerce").dropna()
        if len(values) == 0:
            return 0.0, 0.05
    
        if len(values) < 10:
            tgt = float(values.median())
            spread = float(values.std()) if len(values) > 1 else 0.05
            return tgt, max(spread, 0.03)
    
        q10 = values.quantile(0.10)
        q90 = values.quantile(0.90)
        vf = values[(values >= q10) & (values <= q90)]
    
        if len(vf) == 0:
            vf = values
    
        tgt = float(vf.median())
        spread = float(vf.std()) if len(vf) > 1 else 0.05
        return tgt, max(spread, 0.03)
    
    def score_candidates(cand_gdf, target_ndvi, spread):
        g = cand_gdf.copy()
    
        g["ndvi_dist_n"] = (g["ndvi_mean"] - target_ndvi).abs() / (spread + 1e-6)
        g["ndvi_std_n"] = robust_minmax(g["ndvi_std"])
        g["size_n"] = 1 - robust_minmax(-g["square_size"])  # larger is better
    
        g["score"] = (
            W_NDVI * g["ndvi_dist_n"] +
            W_STD  * g["ndvi_std_n"] -
            W_SIZE * g["size_n"]
        )
    
        g = g.sort_values(["score", "square_size"], ascending=[True, False])
        return g
    
    # ---------------------------------------------------------
    # READ TRUE CLC GEOMETRY
    # ---------------------------------------------------------
    print("Building true class unions from original CLC polygons...", flush=True)
    class_union_gdf, gdf_clc_m = build_class_union_gdf(input_gpkg, CLC_LAYER, clc_crs_target="EPSG:3035")
    
    class_poly_groups = {class_poly_groups.copy()
        for cl, grp in gdf_clc_m.groupby("clc_l3")
    }

    class_union_gdf["clc_l3"] = class_union_gdf["clc_l3"].astype(str)
    class_union_by_class = dict(zip(class_union_gdf["clc_l3"], class_union_gdf.geometry))
    
    print(f"True class unions ready: {len(class_union_by_class)} classes", flush=True)
    
    # ---------------------------------------------------------
    # MAIN SAMPLING
    # ---------------------------------------------------------
    global_selected_geoms = []
    all_selected_rows = []
    season_counts = {}
    
    for season_name in SEASONS_TO_RUN:
        raster_ndvi_path = SEASONAL_NDVI[season_name]
        if not os.path.exists(raster_ndvi_path):
            print(f"⚠️ Missing raster for {season_name}: {raster_ndvi_path}")
            continue
    
        print("\n" + "="*70, flush=True)
        print(f"PROCESSING SEASON: {season_name.upper()}", flush=True)
        print("="*70, flush=True)
    
        with rasterio.open(raster_ndvi_path) as src:
            ndvi_crs = src.crs
            ndvi_nodata = src.nodata
            ndvi_bbox = box(*src.bounds)
    
        # clip true class unions to raster extent
        ndvi_bbox_3035 = gpd.GeoSeries([ndvi_bbox], crs=ndvi_crs).to_crs("EPSG:3035").iloc[0]
    

    
        # -----------------------------------------------------
        # Build season-specific class targets
        # -----------------------------------------------------
        season_class_scored = {}
    
        for (cl, s), cand_gdf in season_candidate_pools.items():
            tgt, spread = season_class_target(cand_gdf["ndvi_mean"])
            scored = score_candidates(cand_gdf, tgt, spread)
            season_class_scored[(cl, s)] = scored
    
        # store counts per season-class
        season_counts[season_name] = {}
    
        # -----------------------------------------------------
        # FAIR CLASS-WISE SELECTION:
        # pick per class, but maintain global no-overlap across seasons
        # -----------------------------------------------------
        class_list = sorted(class_union_by_class.keys(), key=lambda x: (len(x), x))
    
        # We process one class at a time and accept season-specific candidates
        # while respecting global non-overlap.
        for cl in class_list:
            class_selected_local = []
    
            # one pool for this class in this season
            pool = season_class_scored.get((cl, season_name))
            if pool is None or pool.empty:
                season_counts[season_name][cl] = 0
                continue
    
            accepted_rows = []
            for _, row in pool.iterrows():
                geom = row.geometry
                if geom is None or geom.is_empty:
                    continue
    
                if overlaps_too_much(geom, global_selected_geoms, MAX_OVERLAP_FRAC):
                    continue
    
                if overlaps_too_much(geom, class_selected_local, MAX_OVERLAP_FRAC):
                    continue
    
                accepted_rows.append(row)
                class_selected_local.append(geom)
                global_selected_geoms.append(geom)
    
                if len(accepted_rows) >= TARGET_PER_SEASON:
                    break
    
            if accepted_rows:
                accepted_gdf = gpd.GeoDataFrame(accepted_rows, crs="EPSG:3035")
            else:
                accepted_gdf = gpd.GeoDataFrame(columns=list(pool.columns), geometry="geometry", crs="EPSG:3035")
    
            # -------------------------------------------------
            # LAST FILL GRID PASS (for remaining missing samples)
            # -------------------------------------------------
            if len(accepted_gdf) < TARGET_PER_SEASON:
                needed = TARGET_PER_SEASON - len(accepted_gdf)
    
                class_polygon = class_union_by_class.get(cl)
                if class_polygon is not None and not class_polygon.is_empty:
                    try:
                        class_polygon = class_polygon.intersection(ndvi_bbox_3035)
                    except Exception:
                        pass
    
                    class_polygon = keep_valid_geom(class_polygon)
    
                    if class_polygon is not None and not class_polygon.is_empty:
                        # finer grid fallback: force smaller sizes in last fill
                        fallback_sizes = [40, 20]
                        fallback_rows = []
    
                        for square_size in fallback_sizes:
                            if len(fallback_rows) >= needed:
                                break
    
                            raw_squares = generate_grid_candidates(
                                class_polygon=class_polygon,
                                square_size=square_size,
                                max_raw=MAX_RAW_BY_SIZE.get(square_size, 250)
                                #max_raw=MAX_RAW_BY_SIZE[square_size]
                            )
    
                            if len(raw_squares) == 0:
                                continue
    
                            fallback_gdf = gpd.GeoDataFrame(
                                {
                                    "clc_l3": [cl] * len(raw_squares),
                                    "season": [season_name] * len(raw_squares),
                                    "square_size": [square_size] * len(raw_squares),
                                },
                                geometry=raw_squares,
                                crs="EPSG:3035"
                            )
    
                            fallback_gdf = extract_ndvi_stats(fallback_gdf, raster_ndvi_path, ndvi_crs, ndvi_nodata)
                            if fallback_gdf.empty:
                                continue
    
                            tgt, spread = season_class_target(fallback_gdf["ndvi_mean"])
                            fallback_gdf = score_candidates(fallback_gdf, tgt, spread)
    
                            local_now = list(accepted_gdf.geometry)
    
                            for _, row in fallback_gdf.iterrows():
                                geom = row.geometry
                                if geom is None or geom.is_empty:
                                    continue
    
                                if overlaps_too_much(geom, global_selected_geoms, MAX_OVERLAP_FRAC):
                                    continue
    
                                if overlaps_too_much(geom, local_now, MAX_OVERLAP_FRAC):
                                    continue
    
                                fallback_rows.append(row)
                                local_now.append(geom)
                                global_selected_geoms.append(geom)
    
                                if len(fallback_rows) >= needed:
                                    break
    
                        if fallback_rows:
                            fallback_gdf = gpd.GeoDataFrame(fallback_rows, crs="EPSG:3035")
                            accepted_gdf = gpd.GeoDataFrame(
                                pd.concat([accepted_gdf, fallback_gdf], ignore_index=True),
                                crs="EPSG:3035"
                            )
    
            season_counts[season_name][cl] = len(accepted_gdf)
    
            if not accepted_gdf.empty:
                accepted_gdf["sample_source"] = accepted_gdf.get("sample_source", "selected")
                all_selected_rows.append(accepted_gdf)
    
        # season summary
        print(f"Season {season_name} done.", flush=True)
        season_summary = pd.Series(season_counts[season_name]).sort_index()
        print(season_summary, flush=True)
    
    # ---------------------------------------------------------
    # FINAL EXPORT
    # ---------------------------------------------------------
    if len(all_selected_rows) == 0:
        raise ValueError("No samples were selected. Check rasters, CRS and geometry coverage.")
    
    final_samples = gpd.GeoDataFrame(
        pd.concat(all_selected_rows, ignore_index=True),
        crs="EPSG:3035"
    )
    
    # final per season / class
    print("\nTOTAL SAMPLES:", len(final_samples), flush=True)
    print("\nSamples per season:", flush=True)
    print(final_samples["season"].value_counts(), flush=True)
    
    print("\nSamples per class per season:", flush=True)
    pivot = (
        final_samples.groupby(["season", "clc_l3"])
        .size()
        .unstack(fill_value=0)
    )
    print(pivot, flush=True)
    
    # save outputs
    final_out = os.path.join(export_dir, "best_60_per_class_ALL_SEASONS_from_NEW_CELL5.gpkg")
    pivot_csv = os.path.join(export_dir, "samples_per_class_per_season_from_NEW_CELL5.csv")
    final_samples.to_file(final_out, driver="GPKG")
    pivot.to_csv(pivot_csv)
    
    print("Saved final samples:", final_out, flush=True)
    print("Saved pivot:", pivot_csv, flush=True)

In [ ]:
# 7. FINAL PDF GENERATOR — UPDATED FOR NEW SEASONAL OUTPUTS
# ==========================================================

from rasterio.mask import mask as rio_mask
status_area = widgets.Output()
display(status_area)

with StepTimer("Exporting pdf summary..."):

    output_pdf = os.path.join(export_dir, "pdf", "CLC_class_summary_all_seasons.pdf")
    os.makedirs(os.path.dirname(output_pdf), exist_ok=True)

    # ------------------------------------------------------
    # INPUTS
    # ------------------------------------------------------
    merged_gpkg = os.path.join(export_dir, "best_60_per_class_ALL_SEASONS.gpkg")
    assert os.path.exists(merged_gpkg), "Merged seasonal GPKG not found."

    final_all = gpd.read_file(merged_gpkg)

    # ensure season exists
    if "season" not in final_all.columns:
        raise ValueError("❌ 'season' column not found in merged GPKG.")

    # make sure scene metadata exists (if not, create empty fields)
    if "scene_count" not in final_all.columns:
        final_all["scene_count"] = np.nan

    if "scene_dates" not in final_all.columns:
        final_all["scene_dates"] = ""

    # ------------------------------------------------------
    # LOAD ORIGINAL CLC DATA FOR PARENT POLYGONS
    # ------------------------------------------------------
    gdf_clc_all = gdf_clc_m.copy()

    if gdf_clc_all.crs != final_all.crs:
        gdf_clc_all = gdf_clc_all.to_crs(final_all.crs)

    # unions by class for generated samples
    union_by_class = {
        cl: unary_union(gdf_clc_all.loc[gdf_clc_all["clc_l3"] == cl, "geometry"])
        for cl in gdf_clc_all["clc_l3"].unique()
    }

    # original polygons by class + OBJECTID
    class_objectid_lookup = {}
    if "OBJECTID" in gdf_clc_all.columns:
        for cl, g in gdf_clc_all.groupby("clc_l3"):
            class_objectid_lookup[cl] = g.set_index("OBJECTID")

    # ------------------------------------------------------
    # HELPERS
    # ------------------------------------------------------
    def quality_label(score):
        if score >= 80: return "Excellent"
        elif score >= 60: return "Good"
        elif score >= 40: return "Moderate"
        elif score >= 20: return "Low"
        else: return "Poor"

    def quality_color(score):
        if score >= 80: return "green"
        elif score >= 60: return "yellowgreen"
        elif score >= 40: return "orange"
        elif score >= 20: return "orangered"
        else: return "red"

    def put_border(ax, lw=0.8):
        for s in ax.spines.values():
            s.set_visible(True)
            s.set_linewidth(lw)
            s.set_edgecolor("black")

    def wrap_text(text, width=80):
        return "\n".join(textwrap.wrap(str(text), width))

    def get_polygon_in_crs(poly, orig_crs, target_crs):
        return gpd.GeoSeries([poly], crs=orig_crs).to_crs(target_crs).iloc[0]

    def extract_ndvi_vals(poly_src, poly_crs, ndvi_path):
        with rasterio.open(ndvi_path) as src:
            poly_ndvi = get_polygon_in_crs(poly_src, poly_crs, src.crs)
            arr, _ = rio_mask(src, [mapping(poly_ndvi)], crop=True)
            band = arr[0].astype(float)

            if np.ma.isMaskedArray(band):
                vals = band.compressed()
            else:
                vals = band.ravel()

            vals = vals[np.isfinite(vals)]

            nodata = src.nodata
            if nodata is not None:
                vals = vals[vals != nodata]

            vals = vals[(vals >= -1) & (vals <= 1)]
            return vals

    def extract_ndvi_bbox(poly_src, poly_crs, ndvi_path):
        with rasterio.open(ndvi_path) as src:
            poly_ndvi = get_polygon_in_crs(poly_src, poly_crs, src.crs)
            minx, miny, maxx, maxy = poly_ndvi.bounds

            window = rasterio.windows.from_bounds(minx, miny, maxx, maxy, src.transform)
            nd = src.read(1, window=window).astype(float)

            nodata = src.nodata
            if nodata is not None:
                nd[nd == nodata] = np.nan

            affine = src.window_transform(window)
            return nd, affine, src.crs

    def geom_to_crop_pixels(geom_src, geom_crs, affine, raster_crs):
        g = gpd.GeoSeries([geom_src], crs=geom_crs).to_crs(raster_crs).iloc[0]
        inv = ~affine
        segs = []

        if g.geom_type == "Polygon":
            rings = [g.exterior]
        elif g.geom_type == "MultiPolygon":
            rings = [part.exterior for part in g.geoms]
        else:
            return []

        for ring in rings:
            pts = []
            for x, y in ring.coords:
                col, row = inv * (x, y)
                pts.append((col, row))
            segs.append(pts)

        return segs

    def plot_ndvi_hist(ax, vals):
        vals = vals[np.isfinite(vals)]
        ax.hist(vals, bins=40, color="green", alpha=0.7)
        ax.set_title("NDVI distribution")
        ax.set_xlabel("NDVI")
        ax.set_ylabel("Count")

    def get_parent_polygon(row):
        cl = row["clc_l3"]

        # original sample with matching OBJECTID
        if (
            "sample_source" in row.index
            and row["sample_source"] == "original"
            and "OBJECTID" in row.index
            and pd.notna(row["OBJECTID"])
            and cl in class_objectid_lookup
            and row["OBJECTID"] in class_objectid_lookup[cl].index
        ):
            return class_objectid_lookup[cl].loc[row["OBJECTID"], "geometry"]

        # generated sample fallback: use union polygon of the class
        return union_by_class.get(cl, row.geometry)

    # ------------------------------------------------------
    # PDF GENERATION
    # ------------------------------------------------------
    with PdfPages(output_pdf) as pdf:

        # ============================================
        # PAGE 1 — OVERVIEW
        # ============================================
        fig = plt.figure(figsize=(8.27, 11.69))
        ax = fig.add_subplot(111)
        ax.axis("off")

        txt = (
            "CLC Seasonal Sample Quality Report\n\n"
            "This report summarizes the best samples selected from the merged\n"
            "multi-season dataset.\n\n"
            "For each CLC class, the report shows:\n"
            "• season of the selected best sample\n"
            "• sample source (original / generated)\n"
            "• quality score and NDVI statistics\n"
            "• NDVI map of the parent class polygon\n"
            "• NDVI histogram\n"
            "• acquisition dates used for the seasonal NDVI composite\n"
        )

        ax.text(
            0.03, 0.95,
            txt,
            va="top",
            fontsize=12
        )

        pdf.savefig(fig)
        plt.close(fig)

        # ============================================
        # ONE PAGE PER CLASS — BEST SAMPLE ACROSS ALL SEASONS
        # ============================================
        for cl in sorted(final_all["clc_l3"].dropna().unique()):

            g = final_all[final_all["clc_l3"] == cl].copy()

            if len(g) == 0:
                continue

            g = g.sort_values("quality_score", ascending=False)
            best = g.iloc[0]

            season_name = best["season"]
            ndvi_path = SEASONAL_NDVI[season_name]

            if not os.path.exists(ndvi_path):
                print(f"⚠️ NDVI raster not found for season {season_name}: {ndvi_path}")
                continue

            meta = clc_metadata.get(int(cl), {
                "level1": "Unknown",
                "level2": "Unknown",
                "level3": f"CLC {cl}",
                "description": "No metadata available.",
                "example_folder": None
            })

            parent_poly = get_parent_polygon(best)
            sample_poly = best.geometry

            # Example image
            example_img = None
            folder = meta.get("example_folder", None)
            if folder:
                fp = os.path.join(root_dir, "example_images", folder)
                if os.path.exists(fp):
                    jpgs = [f for f in os.listdir(fp) if f.lower().endswith(".jpg")]
                    if jpgs:
                        import matplotlib.image as mpimg
                        example_img = mpimg.imread(os.path.join(fp, jpgs[0]))

            # NDVI extraction
            ndvi_vals = extract_ndvi_vals(parent_poly, final_all.crs, ndvi_path)
            ndvi_img, affine_ndvi, ndvi_crs = extract_ndvi_bbox(parent_poly, final_all.crs, ndvi_path)

            fig = plt.figure(figsize=(8.27, 11.69), constrained_layout=True)
            gs = gridspec.GridSpec(4, 2, figure=fig, height_ratios=[1.4, 2.6, 2.6, 2.2])

            # -------------------------
            # HEADER
            # -------------------------
            ax_header = fig.add_subplot(gs[0, :])
            ax_header.axis("off")

            title = f"CLC Class {cl} — {meta['level3']}"
            subtitle = f"{meta['level1']} | {meta['level2']}"
            desc = meta["description"]

            ax_header.text(0.01, 0.95, wrap_text(title, 60), fontsize=14, fontweight="bold", va="top")
            ax_header.text(0.01, 0.68, wrap_text(subtitle, 60), fontsize=11, style="italic", va="top")
            ax_header.text(0.01, 0.38, wrap_text(desc, 110), fontsize=10, va="top")

            # -------------------------
            # LEFT: EXAMPLE IMAGE
            # -------------------------
            ax_img = fig.add_subplot(gs[1, 0])
            if example_img is not None:
                ax_img.imshow(example_img)
            else:
                ax_img.text(0.5, 0.5, "No example image available", ha="center", va="center")
            ax_img.axis("off")
            put_border(ax_img)

            # -------------------------
            # RIGHT: STATS
            # -------------------------
            ax_stats = fig.add_subplot(gs[1, 1])
            ax_stats.axis("off")

            q = best["quality_score"]
            label = quality_label(q)
            color = quality_color(q)

            stats_text = f"""
Season: {season_name}
Sample source: {best.get('sample_source', 'unknown')}

Quality score: {q:.1f}
Quality class: {label}

NDVI mean: {best.get('ndvi_mean', np.nan):.3f}
NDVI std: {best.get('ndvi_std', np.nan):.3f}
NDVI range: {best.get('ndvi_range', np.nan):.3f}

Interior margin: {best.get('interior_margin_m', np.nan):.1f} m

Scene count: {best.get('scene_count', np.nan)}
Scene dates:
{wrap_text(best.get('scene_dates', ''), 42)}
            """.strip()

            ax_stats.text(
                0.02, 0.98,
                "Sample statistics",
                fontsize=11,
                fontweight="bold",
                va="top"
            )

            ax_stats.text(
                0.02, 0.88,
                stats_text,
                fontsize=10,
                va="top",
                color="black"
            )
            put_border(ax_stats)

            # -------------------------
            # NDVI MAP
            # -------------------------
            gs_maps = gridspec.GridSpecFromSubplotSpec(1, 2, subplot_spec=gs[2, :], wspace=0.15)

            ax_ndvi = fig.add_subplot(gs_maps[0, 0])
            ax_ndvi.imshow(ndvi_img, cmap="RdYlGn")
            ax_ndvi.set_title(f"Seasonal NDVI ({season_name})", fontsize=11)
            ax_ndvi.axis("off")

            for seg in geom_to_crop_pixels(parent_poly, final_all.crs, affine_ndvi, ndvi_crs):
                xs, ys = zip(*seg)
                ax_ndvi.plot(xs, ys, "-r", lw=1.5)

            for seg in geom_to_crop_pixels(sample_poly, final_all.crs, affine_ndvi, ndvi_crs):
                xs, ys = zip(*seg)
                ax_ndvi.plot(xs, ys, "-y", lw=1.5)

            put_border(ax_ndvi)

            # -------------------------
            # RIGHT MAP PANEL:
            # class overview text instead of RGB
            # -------------------------
            ax_txt = fig.add_subplot(gs_maps[0, 1])
            ax_txt.axis("off")
            ax_txt.text(
                0.02, 0.95,
                "Interpretation",
                fontsize=11,
                fontweight="bold",
                va="top"
            )
            ax_txt.text(
                0.02, 0.80,
                "Red outline = original class polygon\n"
                "Yellow outline = selected sample\n\n"
                "This NDVI map corresponds to the season\n"
                "from which the selected sample was ranked.\n\n"
                "The scene dates shown in the statistics panel\n"
                "are the Sentinel-2 acquisitions used to build\n"
                "the seasonal NDVI composite.",
                fontsize=10,
                va="top"
            )
            put_border(ax_txt)

            # -------------------------
            # HISTOGRAM
            # -------------------------
            ax_hist = fig.add_subplot(gs[3, :])
            if len(ndvi_vals) > 0:
                plot_ndvi_hist(ax_hist, ndvi_vals)
            else:
                ax_hist.text(0.5, 0.5, "No NDVI values available", ha="center", va="center")
            put_border(ax_hist)

            pdf.savefig(fig)
            plt.close(fig)

    print("Summary PDF created:", output_pdf)